# Train Eval Experiment 

## Resources

### Import Library

In [8]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# import SKlearn FFNN
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# import custom module
from ffnn import FFNN, Layer

#### Import Dataset

In [9]:
X_train_classification = np.load('dataset/classification/X_train_final.npy')
y_train_classification = np.load('dataset/classification/y_train_final.npy')
X_test_classification = np.load('dataset/classification/X_test_final.npy')
y_test_classification = np.load('dataset/classification/y_test_final.npy')

# Cek shape data
print(f"X_train classification shape    : {X_train_classification.shape}")
print(f"y_train classification shape    : {y_train_classification.shape}")
print(f"X_test classification shape     : {X_test_classification.shape}")
print(f"y_test classification shape     : {y_test_classification.shape}")

X_train_regression = np.load('dataset/regression/X_train_final.npy')
y_train_regression = np.load('dataset/regression/y_train_final.npy')
X_test_regression = np.load('dataset/regression/X_test_final.npy')
y_test_regression = np.load('dataset/regression/y_test_final.npy')

# Cek shape data
print(f"X_train regression shape        : {X_train_regression.shape}")
print(f"y_train regression shape        : {y_train_regression.shape}")
print(f"X_test regression shape         : {X_test_regression.shape}")
print(f"y_test regression shape         : {y_test_regression.shape}")

X_train classification shape    : (9844, 21)
y_train classification shape    : (9844,)
X_test classification shape     : (2000, 21)
y_test classification shape     : (2000,)
X_train regression shape        : (8000, 20)
y_train regression shape        : (8000, 1)
X_test regression shape         : (2000, 20)
y_test regression shape         : (2000, 1)


## Experiment

### Calssification

#### Arsitektur 1

Input (21)
Hidden 1 (32)
Hidden 2 (16)
Output (1)

##### SKLearn

In [10]:
# activations test
sklearn_activations = ['identity', 'relu', 'logistic', 'tanh']
sklearn_models = {}

for activation in sklearn_activations:
    mlp = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation=activation,   
        solver='sgd',
        learning_rate_init=0.01,
        max_iter=50,
        batch_size=32,
        random_state=42
    )
    mlp.fit(X_train_classification, y_train_classification)
    sklearn_models[activation] = mlp
    print(f"Finish model {activation}")

Finish model identity


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Finish model relu


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Finish model logistic
Finish model tanh


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


In [11]:
# Evaluation
for activation, model in sklearn_models.items():
    y_pred = model.predict(X_test_classification)
    acc = accuracy_score(y_test_classification, y_pred)
    print(f"Accuracy for {activation}: {acc:.4f}")
    print(f"Classification Report for {activation}:\n{classification_report(y_test_classification, y_pred)}")
    print(f"Confusion Matrix for {activation}:\n{confusion_matrix(y_test_classification, y_pred)}\n")
    

Accuracy for identity: 0.7355
Classification Report for identity:
              precision    recall  f1-score   support

           0       0.65      0.66      0.66       769
           1       0.79      0.78      0.78      1231

    accuracy                           0.74      2000
   macro avg       0.72      0.72      0.72      2000
weighted avg       0.74      0.74      0.74      2000

Confusion Matrix for identity:
[[510 259]
 [270 961]]

Accuracy for relu: 0.7165
Classification Report for relu:
              precision    recall  f1-score   support

           0       0.64      0.60      0.62       769
           1       0.76      0.79      0.77      1231

    accuracy                           0.72      2000
   macro avg       0.70      0.70      0.70      2000
weighted avg       0.71      0.72      0.71      2000

Confusion Matrix for relu:
[[465 304]
 [263 968]]

Accuracy for logistic: 0.7345
Classification Report for logistic:
              precision    recall  f1-score   supp

##### Custom FFNN

In [12]:
model = FFNN(loss='bce')
model.add(Layer(input_size=21, output_size=32, activation='relu', init_method='he'))
model.add(Layer(input_size=32, output_size=16, activation='relu', init_method='he'))
model.add(Layer(input_size=16, output_size=1, activation='sigmoid', init_method='xavier'))

training = model.fit(
    X_train_classification,
    y_train_classification.reshape(-1,1),
    epochs=50,
    batch_size=32, 
    learning_rate=0.01, 
    optimizer='adam',
    validation_data=(X_test_classification, y_test_classification.reshape(-1, 1))
)

y_pred_prob = model.predict(X_test_classification)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
print(f"\nAccuracy: {accuracy_score(y_test_classification, y_pred):.4f}")
print(classification_report(y_test_classification, y_pred))

epoch 1/50 - loss: 0.4928 - val_loss: 0.5295
epoch 2/50 - loss: 0.4449 - val_loss: 0.5131
epoch 3/50 - loss: 0.4299 - val_loss: 0.5202
epoch 4/50 - loss: 0.4218 - val_loss: 0.5192
epoch 5/50 - loss: 0.4204 - val_loss: 0.5135
epoch 6/50 - loss: 0.4148 - val_loss: 0.5145
epoch 7/50 - loss: 0.4111 - val_loss: 0.5150
epoch 8/50 - loss: 0.4089 - val_loss: 0.5452
epoch 9/50 - loss: 0.4078 - val_loss: 0.5360
epoch 10/50 - loss: 0.4057 - val_loss: 0.5336
epoch 11/50 - loss: 0.4032 - val_loss: 0.5377
epoch 12/50 - loss: 0.4025 - val_loss: 0.5342
epoch 13/50 - loss: 0.3980 - val_loss: 0.5458
epoch 14/50 - loss: 0.3978 - val_loss: 0.5451
epoch 15/50 - loss: 0.3950 - val_loss: 0.5476
epoch 16/50 - loss: 0.3900 - val_loss: 0.5536
epoch 17/50 - loss: 0.3902 - val_loss: 0.5619
epoch 18/50 - loss: 0.3914 - val_loss: 0.5598
epoch 19/50 - loss: 0.3892 - val_loss: 0.5553
epoch 20/50 - loss: 0.3846 - val_loss: 0.5584
epoch 21/50 - loss: 0.3844 - val_loss: 0.5616
epoch 22/50 - loss: 0.3803 - val_loss: 0.58

### Regresi

#### Arsitektur 1

##### SKLearn

In [13]:
from sklearn.neural_network import MLPRegressor

# activations test
sklearn_models_reg = {}

for activation in sklearn_activations:
    mlp_reg = MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation=activation,   
        solver='adam',
        learning_rate_init=0.01,
        max_iter=100,
        batch_size=32,
        random_state=42
    )
    mlp_reg.fit(X_train_regression, y_train_regression)
    sklearn_models_reg[activation] = mlp_reg
    print(f"Finish model {activation}")

/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1645: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Finish model identity


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1645: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1645: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Finish model relu


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1645: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Finish model logistic
Finish model tanh


/home/astha/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [16]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
# evaluasi

for activation in sklearn_activations:
    y_pred = sklearn_models_reg[activation].predict(X_test_regression)
    mse = mean_squared_error(y_test_regression, y_pred)
    r2 = r2_score(y_test_regression, y_pred)
    print(f"SKLearn Regression {activation}         - MSE: {mse:.4f}, R2 Score: {r2:.4f}")

SKLearn Regression identity         - MSE: 2.2553, R2 Score: 0.0210
SKLearn Regression relu         - MSE: 2.5995, R2 Score: -0.1285
SKLearn Regression logistic         - MSE: 2.7992, R2 Score: -0.2152
SKLearn Regression tanh         - MSE: 2.8653, R2 Score: -0.2439
